# AxonScope Colab Double-Cable End-To-End

Use this notebook in a Google Colab GPU runtime after the isolated linear-solver benchmark. It clones `bench-colab`, installs AxonScope, runs end-to-end double-cable batch-kernel cases with selected recording and `Iinj` modes, optionally runs a forced CPU comparison, zips the result folder, and downloads it through the browser.

Local setup before running this notebook:

```bash
git add -A
git commit -m "Benchmark double-cable end-to-end"
make bench-colab-push
```

In [ ]:
import csv
import datetime
import json
import os
import pathlib
import shutil
import subprocess

from google.colab import files

REPO_URL = "https://github.com/louisreg/AxonScope.git"
BRANCH = "bench-colab"

# Cases:
# - smoke: tiny sanity run.
# - gpu_recording_matrix: compare none/center/full recording pressure.
# - gpu_iinj_matrix: compare absent, dense-zero, and nonzero Iinj pressure.
# - trace_observer_auto: focused trace for observer-only auto path.
CASE = "gpu_recording_matrix"

RUN_GPU = True
RUN_CPU = False
TRACE_GPU = False
TRACE_CPU = False
CREATE_PERFETTO = False

CASES = {
    "smoke": {
        "batch_sizes": [2],
        "nx": [51],
        "nt": [3],
        "dt": 0.05,
        "recordings": ["center"],
        "iinj_modes": ["none"],
        "solvers": ["thomas"],
        "warmups": 0,
        "repeats": 1,
    },
    "gpu_recording_matrix": {
        "batch_sizes": [512, 1024, 2048],
        "nx": [51, 64, 96],
        "nt": [500, 1000],
        "dt": 0.01,
        "recordings": ["none", "center", "full"],
        "iinj_modes": ["none"],
        "solvers": ["auto", "thomas", "pcr_adaptive"],
        "warmups": 1,
        "repeats": 3,
    },
    "gpu_iinj_matrix": {
        "batch_sizes": [512, 1024, 2048],
        "nx": [51, 64],
        "nt": [500],
        "dt": 0.01,
        "recordings": ["none", "center"],
        "iinj_modes": ["none", "dense_zero", "nonzero"],
        "solvers": ["auto", "pcr_adaptive"],
        "warmups": 1,
        "repeats": 3,
    },
    "trace_observer_auto": {
        "batch_sizes": [1024],
        "nx": [64],
        "nt": [1000],
        "dt": 0.01,
        "recordings": ["none"],
        "iinj_modes": ["none"],
        "solvers": ["auto"],
        "warmups": 1,
        "repeats": 1,
        "trace_gpu": True,
    },
}

PKG_DIR = pathlib.Path("/content/AxonScope")
RESULTS_ROOT = PKG_DIR / "benchmark/results/solvers"


def run_command(command, *, cwd=None, env=None):
    print("\n$", " ".join(str(part) for part in command), flush=True)
    subprocess.run(command, cwd=cwd, env=env, check=True)


def setup_repo():
    if PKG_DIR.exists():
        shutil.rmtree(PKG_DIR)
    run_command(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PKG_DIR)])
    run_command(["git", "rev-parse", "--short", "HEAD"], cwd=PKG_DIR)
    run_command(["python", "-m", "pip", "install", "-U", "pip"])
    run_command(["python", "-m", "pip", "install", "-e", ".[benchmark]"], cwd=PKG_DIR)
    run_command(["bash", "-lc", "nvidia-smi || true"], cwd=PKG_DIR)


def verify_backend(label, *, env=None):
    expected = "gpu" if label == "gpu" else "cpu"
    code = f"""
import jax
backend = jax.default_backend()
print('jax backend:', backend)
print('jax devices:', jax.devices())
if backend != {expected!r}:
    raise SystemExit(f'Expected JAX backend {expected!r}, got {{backend!r}}.')
"""
    run_command(["python", "-c", code], cwd=PKG_DIR, env=env)


def benchmark_command(config, *, label, out_dir, trace=False):
    command = [
        "python",
        "benchmark/solvers/bench_double_cable_end_to_end.py",
        "--batch-sizes",
        *[str(value) for value in config["batch_sizes"]],
        "--nx",
        *[str(value) for value in config["nx"]],
        "--nt",
        *[str(value) for value in config["nt"]],
        "--dt",
        str(config["dt"]),
        "--recordings",
        *config["recordings"],
        "--iinj-modes",
        *config["iinj_modes"],
        "--solvers",
        *config["solvers"],
        "--warmups",
        str(config["warmups"]),
        "--repeats",
        str(config["repeats"]),
        "--out-dir",
        str(out_dir),
        "--prefix",
        label,
    ]
    if trace:
        command.extend(["--jax-trace", "--jax-trace-dir", str(out_dir / label / "jax_traces")])
        if CREATE_PERFETTO:
            command.append("--jax-trace-create-perfetto")
    return command


def run_label(label, config, *, out_dir, env=None, trace=False):
    verify_backend(label, env=env)
    run_command(benchmark_command(config, label=label, out_dir=out_dir, trace=trace), cwd=PKG_DIR, env=env)
    return out_dir / label / "summary.csv"


def print_summary(label, csv_path, *, top_n=12):
    with csv_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    print(f"\n{label.upper()} summary: {csv_path}")
    for row in rows[:top_n]:
        kernel_ms = float(row["kernel_enqueue_median_ms"]) + float(row["kernel_wait_median_ms"])
        print(
            f"{row['requested_solver']}->{row['resolved_solver']} "
            f"B={row['batch_size']} Nx={row['actual_nx']} Nt={row['nt']} "
            f"rec={row['recording']} iinj={row['iinj_mode']}: "
            f"kernel {kernel_ms:.3f} ms | total+inputs {float(row['total_with_inputs_ms']):.3f} ms"
        )
    if len(rows) > top_n:
        print(f"... {len(rows) - top_n} more rows")


setup_repo()
if CASE not in CASES:
    raise ValueError(f"Unknown CASE={CASE!r}; choose one of {sorted(CASES)}")
config = dict(CASES[CASE])
run_id = f"colab_double_cable_e2e_{CASE}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
out_dir = RESULTS_ROOT / run_id
out_dir.mkdir(parents=True, exist_ok=True)

print("Selected end-to-end benchmark case:", CASE)
print(json.dumps(config, indent=2, sort_keys=True))

if RUN_GPU:
    gpu_trace = TRACE_GPU or bool(config.get("trace_gpu", False))
    gpu_csv = run_label("gpu", config, out_dir=out_dir, trace=gpu_trace)
    print_summary("gpu", gpu_csv)

if RUN_CPU:
    cpu_env = os.environ.copy()
    cpu_env["JAX_PLATFORMS"] = "cpu"
    cpu_trace = TRACE_CPU or bool(config.get("trace_cpu", False))
    cpu_csv = run_label("cpu", config, out_dir=out_dir, env=cpu_env, trace=cpu_trace)
    print_summary("cpu", cpu_csv)

archive_path = shutil.make_archive(str(out_dir), "zip", out_dir)
print(f"\nResults folder: {out_dir}")
print(f"Archive: {archive_path}")
files.download(archive_path)
